# Phase 5 · Manuscript figures (R1 revision)

Produces the figures of the revised manuscript from the canonical v2 results (Phase 3) and the Phase 4 tables.
Conventions (D-20): no title inside a figure, panel labels (a), (b), ... below the x-axis, PNG + PDF at 300 dpi, the data of every figure as CSV. Output: `fig/manuscript/`. A caption draft for every figure is written to `fig/manuscript/captions.md`.

| Figure | Content |
|---|---|
| Fig 1 | framework diagram (`Fig01_framework.drawio`, editable; export PNG/PDF from draw.io) |
| Fig 2 | classifiers: ROC, calibration, normalisation ablation (epochs) |
| Fig 3 | XFI against severity with cross-patient reference (A, B) |
| Fig 4 | paired Grad-CAM minus Eigen-CAM differences (all models) |
| Fig 5 | localization of TP patients under perturbation (A, B) |
| Fig 6 | background activation with absolute threshold (A, B) |
| Fig 7 | validity: input PSNR and localization change |
| Fig 8 | baselines (all models) |
| Fig 9 | prediction flips |
| Fig 10 | representative heatmaps with centre-of-mass drift (Model A) |
| Fig 11 | centre-of-mass drift, quantitative (A, B) |
| Fig S1-S3 | Fig 3, 5, 6 for all four models |
| Fig S4 | Eigen-CAM sign ablation |
| Fig S5 | representative heatmaps (Model B) |

## Configuration cell

In [1]:
import sys, json, shutil
from pathlib import Path
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from r1lib import style as S, stats as ST
S.apply_style()
T4 = ROOT / "output" / "phase4"; P3 = ROOT / "output" / "phase3"; P2 = ROOT / "output" / "phase2"
FIG = ROOT / "fig" / "manuscript"; FIG.mkdir(parents=True, exist_ok=True)
ORDER = ["A", "Arep", "Anorm", "B"]
ALL = [m for m in ORDER if m in set(pd.read_csv(T4 / "R3_xfi_by_condition.csv").model)]
MAIN = [m for m in ["A", "B"] if m in ALL]
LAB = {"A": "Model A", "Arep": "Model A-rep", "Anorm": "Model A-norm", "B": "Model B"}
LS = {"A": "-", "Arep": "--", "Anorm": ":", "B": "-."}
SHORT = {"A": "A", "Arep": "A-rep", "Anorm": "A-norm", "B": "B"}
def group_axis(a):
    """Ticks L1-L3 per perturbation type, type names below (9 conditions on one axis)."""
    a.set_xticks(range(9), [f"L{l}" + (f"\n{S.TYPE_LABELS[t]}" if l == 2 else "") for t in TYPES for l in LEVELS], fontsize=7)
    for v in (2.5, 5.5): a.axvline(v, color=S.GRID, lw=0.8)
    a.grid(axis="x", visible=False)
METHODS, TYPES, LEVELS = ["GradCAM", "EigenCAM"], S.TYPES, [1, 2, 3]
CAPTIONS = {}
def save(fig, stem, data, caption):
    S.save_figure(fig, stem, data, FIG); CAPTIONS[stem] = caption; print("saved", stem)
R = {k: pd.read_csv(T4 / f"{k}.csv") for k in ["R1_classifiers", "R2_grad_vs_eigen", "R3_xfi_by_condition", "R6_tp_iou_under_perturbation",
                                              "R7_absolute_false_activation", "R8_validity", "R10_eigen_sign", "R11_baselines", "R12_flips"]}
cross = R["R11_baselines"][R["R11_baselines"].baseline.str.startswith("CrossPatient")].assign(
    method=lambda d: d.baseline.str.split("_").str[1]).set_index(["model", "method"]).mean_xfi
print("models:", ALL, "| main text:", MAIN)

models: ['A', 'Arep', 'Anorm', 'B'] | main text: ['A', 'B']


## Fig 2 · Classifiers

In [2]:
from sklearn.metrics import roc_curve, roc_auc_score
y_df = pd.read_csv(P3 / "xfi_A" / "results_main.csv", usecols=["Patient", "Label", "prob_clean"]).drop_duplicates("Patient").set_index("Patient")
probs = {"A": y_df.prob_clean}
for M in ["Arep", "Anorm", "B"]:
    f = P2 / f"train_{M}" / "test_probs.csv"
    if f.exists(): probs[M] = pd.read_csv(f).set_index("patientId").prob
y = y_df.Label.reindex(y_df.index).values.astype(int); cls = [m for m in ORDER if m in probs]
fig, ax = plt.subplots(2, 2, figsize=(7.2, 6.2)); d_roc, d_rel = [], []
for M in cls:
    p = probs[M].reindex(y_df.index).values; fpr, tpr, _ = roc_curve(y, p)
    ax[0, 0].plot(fpr, tpr, ls=LS[M], color=S.MODEL_COLORS[M], lw=1.6, label=f"{SHORT[M]} (AUC {roc_auc_score(y, p):.3f})")
    rel = ST.reliability_curve(y, p, 10); ax[0, 1].plot(rel.mean_pred, rel.frac_pos, ls=LS[M], color=S.MODEL_COLORS[M], marker=S.MODEL_MARKERS[M], ms=3.5, lw=1.4,
                                                       label=f"{SHORT[M]} (ECE {ST.ece_score(y, p):.3f})")
    d_roc.append(pd.DataFrame({"model": M, "fpr": fpr, "tpr": tpr})); d_rel.append(rel.assign(model=M))
for a in ax[0]: a.plot([0, 1], [0, 1], color=S.NEUTRAL, lw=0.8, ls="--"); a.set(xlim=(0, 1), ylim=(0, 1.01))
ax[0, 0].set(xlabel="False positive rate", ylabel="True positive rate"); ax[0, 0].legend(loc="lower right", fontsize=7)
ax[0, 1].set(xlabel="Mean predicted probability", ylabel="Observed fraction positive"); ax[0, 1].legend(loc="upper left", fontsize=7)
ep = []
for M in ["Arep", "Anorm"]:
    f = P2 / f"train_{M}" / "epochs.csv"
    if f.exists():
        e = pd.read_csv(f); ep.append(e.assign(model=M))
        ax[1, 0].plot(e.epoch, e.val_auc, ls=LS[M], color=S.MODEL_COLORS[M], marker=S.MODEL_MARKERS[M], ms=4.5, lw=1.6, label=LAB[M] + (" (ImageNet norm.)" if M == "Anorm" else " (no norm.)"))
        ax[1, 1].plot(e.epoch, e.val_ece, ls=LS[M], color=S.MODEL_COLORS[M], marker=S.MODEL_MARKERS[M], ms=4.5, lw=1.6)
ax[1, 0].set(xlabel="Epoch", ylabel="Validation AUC"); ax[1, 1].set(xlabel="Epoch", ylabel="Validation ECE", ylim=(0, None))
for a in ax[1]: a.set_xticks([1, 2, 3, 4, 5])
ax[1, 1].legend(*ax[1, 0].get_legend_handles_labels(), loc="upper left", fontsize=7)
for a, l in zip(ax.ravel(), "abcd"): S.panel_label(a, l)
fig.subplots_adjust(hspace=0.55, wspace=0.3)
save(fig, "Fig02_classifiers", {"roc": pd.concat(d_roc), "reliability": pd.concat(d_rel), "epochs": pd.concat(ep) if ep else pd.DataFrame()},
     "Classifier performance on the 5,337 test patients. (a) ROC curves and (b) reliability diagrams (10 equal-width bins) of Model A (the classifier of the submitted study), "
     "Model A-rep (same recipe, retrained), Model A-norm (same recipe with ImageNet normalisation) and Model B (ImageNet-pretrained, class-weighted loss). "
     "(c) Validation AUC and (d) validation expected calibration error per epoch for Model A-rep and Model A-norm, which differ only in input normalisation.")

saved Fig02_classifiers


In [3]:
def grid(models, value_fn, ylabel, stem, data, caption, legend_extra=None, ylim0=True, xticks=("L1", "L2", "L3"), xs=None):
    nr = len(models); fig, ax = plt.subplots(nr, 3, figsize=(7.2, 2.25 * nr), sharey=True, squeeze=False); letters = iter("abcdefghijklmnop")
    for i, M in enumerate(models):
        for j, t in enumerate(TYPES):
            a = ax[i, j]
            for m in METHODS:
                x, yv, lo, hi = value_fn(M, m, t)
                a.fill_between(x, lo, hi, color=S.METHOD_COLORS[m], alpha=0.18, lw=0)
                a.plot(x, yv, marker=S.METHOD_MARKERS[m], color=S.METHOD_COLORS[m], ms=4.5, lw=1.8)
                if legend_extra == "cross": a.axhline(cross.loc[(M, m)], color=S.METHOD_COLORS[m], ls="--", lw=1)
            a.set_xticks(xs if xs is not None else LEVELS, xticks); a.grid(axis="x", visible=False)
            a.set_xlabel(S.TYPE_LABELS[t]); S.panel_label(a, next(letters))
        ax[i, 0].set_ylabel(f"{LAB[M]}\n{ylabel}")
    if ylim0: ax[0, 0].set_ylim(0, None)
    h = [Line2D([], [], color=S.METHOD_COLORS[m], marker=S.METHOD_MARKERS[m], label=S.METHOD_LABELS[m]) for m in METHODS]
    if legend_extra == "cross": h.append(Line2D([], [], color=S.NEUTRAL, ls="--", lw=1, label="cross-patient XFI"))
    fig.legend(handles=h, loc="upper center", ncol=len(h), bbox_to_anchor=(0.5, 0.995), fontsize=7.5)
    fig.subplots_adjust(top=1 - 0.2 / nr)
    fig.subplots_adjust(wspace=0.1, hspace=0.8)
    save(fig, stem, data, caption)

def kin(M, m, t):
    s = R["R3_xfi_by_condition"]; s = s[(s.model == M) & (s.method == m) & (s.type == t)].sort_values("level")
    return s.level, s.mean_xfi, s.ci_low, s.ci_high
def loc(M, m, t):
    s = R["R6_tp_iou_under_perturbation"]; s = s[(s.model == M) & (s.method == m) & (s.type == t)].sort_values("level")
    return (np.r_[0, s.level], np.r_[s.clean.iloc[0], s.noisy], np.r_[s.clean_ci_low.iloc[0], s.noisy_ci_low], np.r_[s.clean_ci_high.iloc[0], s.noisy_ci_high])
def far(M, m, t):
    s = R["R7_absolute_false_activation"]; s = s[(s.model == M) & (s.method == m) & (s.type == t)].sort_values("level")
    return s.level, s.far_bg_abs5_pos, s.far_bg_abs5_pos_ci_low, s.far_bg_abs5_pos_ci_high

## Fig 3, Fig 5, Fig 6 (main text: A and B) and Fig S1-S3 (all models)

In [4]:
cap3 = ("Mean XFI against perturbation severity for Grad-CAM and Eigen-CAM. Rows: classifiers; columns: Gaussian noise (sigma = 0.1, 0.2, 0.3), Gaussian blur "
        "(kernel 3, 5, 7) and salt-and-pepper noise (1, 2, 3 % of pixels). Shaded bands: 95 % patient-bootstrap confidence intervals. Dashed lines: cross-patient XFI "
        "of the same method and classifier, i.e. the typical dissimilarity between the clean explanations of two different patients.")
cap5 = ("Overlap (Rel-90 IoU) between the explanation and the radiologist's bounding box for correctly classified positive patients (true positives), on the clean "
        "image and under each perturbation level. Shaded bands: 95 % patient-bootstrap confidence intervals.")
cap6 = ("Background activation of positive patients: share of pixels outside the bounding box with normalised activation >= 0.5, on the clean image and under each "
        "perturbation level. This absolute-threshold measure replaces the percentile-based false activation rate of the original submission. Shaded bands: 95 % CIs.")
grid(MAIN, kin, "Mean XFI", "Fig03_kinetics", R["R3_xfi_by_condition"].query("model in @MAIN"), cap3, legend_extra="cross")
grid(MAIN, loc, "IoU with lesion box", "Fig05_localization", R["R6_tp_iou_under_perturbation"].query("model in @MAIN"), cap5, xticks=("clean", "L1", "L2", "L3"), xs=[0, 1, 2, 3])
grid(MAIN, far, "Background share ≥ 0.5", "Fig06_background_activation", R["R7_absolute_false_activation"].query("model in @MAIN"), cap6,
     xticks=("clean", "L1", "L2", "L3"), xs=[0, 1, 2, 3])
if len(ALL) > len(MAIN):
    grid(ALL, kin, "Mean XFI", "FigS1_kinetics_all_models", R["R3_xfi_by_condition"], cap3.replace("Rows: classifiers", "Rows: all four classifiers"), legend_extra="cross")
    grid(ALL, loc, "IoU with lesion box", "FigS2_localization_all_models", R["R6_tp_iou_under_perturbation"], cap5 + " All four classifiers.", xticks=("clean", "L1", "L2", "L3"), xs=[0, 1, 2, 3])
    grid(ALL, far, "Background share ≥ 0.5", "FigS3_background_activation_all_models", R["R7_absolute_false_activation"], cap6 + " All four classifiers.",
         xticks=("clean", "L1", "L2", "L3"), xs=[0, 1, 2, 3])

saved Fig03_kinetics


saved Fig05_localization


saved Fig06_background_activation


saved FigS1_kinetics_all_models


saved FigS2_localization_all_models


saved FigS3_background_activation_all_models


## Fig 4 · Paired differences Grad-CAM minus Eigen-CAM

In [5]:
r2 = R["R2_grad_vs_eigen"]; nc = 2 if len(ALL) > 2 else len(ALL); nr = int(np.ceil(len(ALL) / nc))
fig, ax = plt.subplots(nr, nc, figsize=(7.2, 3.4 * nr), sharey=True, squeeze=False); ax = ax.ravel(); f4 = []
lim = max(r2.median_ci_low.abs().max(), r2.median_ci_high.abs().max()) * 1.1
for a, M, l in zip(ax, ALL, "abcd"):
    d = r2[r2.model == M].copy(); d["row"] = np.where(d.type == "all", "All conditions", d.type.map(S.TYPE_LABELS) + ", L" + d.level.astype(str))
    d = d.iloc[::-1].reset_index(drop=True); f4.append(d)
    for i, r in d.iterrows():
        al = r.type == "all"
        p_ = r.p if al else r.p_holm
        col = S.NEUTRAL if p_ >= 0.05 else (S.METHOD_COLORS["GradCAM"] if r.median_diff < 0 else S.METHOD_COLORS["EigenCAM"])
        a.errorbar(r.median_diff, i, xerr=[[r.median_diff - r.median_ci_low], [r.median_ci_high - r.median_diff]], fmt="D" if al else "o", ms=5.5 if al else 5,
                   color=col, mfc=col, mec="white" if not al else col, capsize=2, lw=1.3)
    a.axvline(0, color=S.INK, lw=0.9); a.grid(axis="y", visible=False); a.set_xlim(-lim, lim)
    a.set_yticks(range(len(d)), d.row); a.set_xlabel(f"{LAB[M]}\nΔXFI (Grad − Eigen)"); S.panel_label(a, l)
fig.legend(handles=[Line2D([], [], color=S.METHOD_COLORS["GradCAM"], marker="o", ls="", label="Grad-CAM less fragile"),
                    Line2D([], [], color=S.METHOD_COLORS["EigenCAM"], marker="o", ls="", label="Eigen-CAM less fragile"),
                    Line2D([], [], color=S.NEUTRAL, marker="o", ls="", label="not significant (Holm)"),
                    Line2D([], [], color=S.INK2, marker="D", ls="", label="all conditions")], loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.0), fontsize=7.5)
fig.subplots_adjust(wspace=0.08, hspace=0.45, top=0.94)
save(fig, "Fig04_grad_vs_eigen", pd.concat(f4).drop(columns="row"),
     "Median paired difference in XFI between Grad-CAM and Eigen-CAM (Grad-CAM minus Eigen-CAM) per perturbation condition and for the patient-level mean over all "
     "conditions, for each classifier. Bars: 95 % patient-bootstrap CIs. Values below zero: Grad-CAM less fragile. Colour: method with the lower XFI where the paired Wilcoxon test is "
     "significant after Holm correction (overall comparison: uncorrected); grey: not significant.")

saved Fig04_grad_vs_eigen


## Fig 7 · Validity

In [6]:
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.0)); d7a, d7b = [], []; bins = np.arange(9, 43, 3)
for M in ALL:
    f = P3 / f"xfi_{M}" / "results_main.csv"
    if not f.exists(): continue
    x = pd.read_csv(f, usecols=["Patient", "Label", "Method", "xfi", "input_psnr", "iou_rel90_c", "iou_rel90_n"])
    x = x[x.Method.isin(METHODS)]; x["abs_d_iou"] = (x.iou_rel90_c - x.iou_rel90_n).abs()
    for m in METHODS:
        s = x[x.Method == m].copy(); s["bin"] = pd.cut(s.input_psnr, bins)
        g = s.groupby("bin", observed=True).agg(xfi=("xfi", "mean"), n=("xfi", "size"), psnr=("input_psnr", "mean")).reset_index(); g = g[g.n >= 50]
        ax[0].plot(g.psnr, g.xfi, ls=LS[M], color=S.METHOD_COLORS[m], marker=S.METHOD_MARKERS[m], ms=3, lw=1.2)
        d7a.append(g.drop(columns="bin").assign(model=M, method=m))
        sp = s[s.Label == 1].copy(); sp["dec"] = pd.qcut(sp.xfi, 10, labels=False, duplicates="drop")
        h = sp.groupby("dec").agg(abs_d_iou=("abs_d_iou", "mean"), xfi=("xfi", "mean")).reset_index()
        ax[1].plot(h.dec + 1, h.abs_d_iou, ls=LS[M], color=S.METHOD_COLORS[m], marker=S.METHOD_MARKERS[m], ms=3, lw=1.2); d7b.append(h.assign(model=M, method=m))
ax[0].set(xlabel="Input PSNR, clean vs perturbed (dB)", ylabel="Mean XFI", ylim=(0, None))
ax[1].set(xlabel="XFI decile (1 = least fragile)", ylabel="Mean |ΔIoU| (Rel-90)", ylim=(0, None)); ax[1].set_xticks(range(1, 11))
mods = [m for m in ALL if (P3 / f"xfi_{m}" / "results_main.csv").exists()]
h = [Line2D([], [], color=S.METHOD_COLORS[m], marker=S.METHOD_MARKERS[m], label=S.METHOD_LABELS[m]) for m in METHODS] + \
    [Line2D([], [], color=S.INK, ls=LS[M], label=LAB[M]) for M in mods]
fig.legend(handles=h, loc="upper center", ncol=len(h), bbox_to_anchor=(0.5, 1.06), fontsize=7, handlelength=3.2)
S.panel_label(ax[0], "a"); S.panel_label(ax[1], "b"); fig.subplots_adjust(wspace=0.3)
r8 = R["R8_validity"]
save(fig, "Fig07_validity", {"psnr_bins": pd.concat(d7a), "xfi_deciles": pd.concat(d7b)},
     "XFI follows the degradation of the input and the change in localization. (a) Mean XFI by bin of input PSNR (clean against perturbed image, 3 dB bins, all "
     f"conditions pooled); Spearman correlation between XFI and input PSNR ranges from {r8.rho_input_psnr.max():.2f} to {r8.rho_input_psnr.min():.2f}. "
     "(b) Mean absolute change in Rel-90 IoU by XFI decile, positive patients; Spearman correlation "
     f"{r8.rho_abs_d_iou_pos.min():.2f} to {r8.rho_abs_d_iou_pos.max():.2f}. Colour: explanation method; line style: classifier.")

saved Fig07_validity


## Fig 8 · Baselines

In [7]:
r11 = R["R11_baselines"]; fig, ax = plt.subplots(1, len(ALL), figsize=(7.2 if len(ALL) <= 2 else 9.0, 3.1), sharey=True); ax = np.atleast_1d(ax)
bl = {"CrossPatient": ("--", "cross-patient"), "ModelRandom": (":", "randomised model"), "Random_Smooth": ("-.", "smooth random"), "Random_Pixel": ("-", "pixel random")}
d8 = []
for a, M, l in zip(ax, ALL, "abcd"):
    f = P3 / f"xfi_{M}" / "results_main.csv"
    pos = 0; ticks = []
    if f.exists():
        x = pd.read_csv(f, usecols=["Patient", "Method", "Level", "xfi"])
        for lv in LEVELS:
            for m in METHODS:
                v = x[(x.Method == m) & (x.Level == lv)].groupby("Patient").xfi.mean().values
                bp = a.boxplot(v, positions=[pos], widths=0.6, showfliers=False, patch_artist=True)
                [p.set(facecolor="white", edgecolor=S.METHOD_COLORS[m], lw=1.2) for p in bp["boxes"]]
                [p.set(color=S.METHOD_COLORS[m], lw=1.1) for k in ["whiskers", "caps", "medians"] for p in bp[k]]
                d8.append(pd.DataFrame({"model": M, "method": m, "level": lv, "xfi": v})); pos += 1
            ticks.append(pos - 1.5); pos += 0.6
    for k, (ls, lab) in bl.items():
        a.axhline(r11[(r11.model == M) & r11.baseline.str.startswith(k)].mean_xfi.mean(), color=S.NEUTRAL, ls=ls, lw=1.1)
    a.set_xticks(ticks, ["L1", "L2", "L3"][:len(ticks)]); a.grid(axis="x", visible=False); a.set_xlabel(LAB[M]); S.panel_label(a, l)
ax[0].set_ylabel("Patient-level XFI"); ax[0].set_ylim(0, 0.8)
h = [Line2D([], [], color=S.METHOD_COLORS[m], lw=2, label=S.METHOD_LABELS[m]) for m in METHODS] + [Line2D([], [], color=S.NEUTRAL, ls=ls, lw=1.1, label=lab) for ls, lab in bl.values()]
fig.legend(handles=h, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.07), fontsize=7)
fig.subplots_adjust(wspace=0.08)
save(fig, "Fig08_baselines", {"methods": pd.concat(d8) if d8 else pd.DataFrame(), "baselines": r11},
     "Patient-level XFI (mean over the three perturbation types) per severity level for Grad-CAM and Eigen-CAM, against baselines (horizontal lines): explanations of "
     "two different patients (cross-patient), explanations of a copy of the classifier whose last residual block and output layer were re-initialised (randomised "
     "model), two independent smooth random maps (7 x 7 upsampled) and two independent pixel-noise maps. Boxes: interquartile range and median; whiskers: 1.5 IQR.")

saved Fig08_baselines


## Fig 9 · Prediction flips

In [8]:
r12 = R["R12_flips"]; fr = r12[r12.kind == "flip_rate"].assign(level=lambda d: d.level.astype(int)); xx = np.arange(9); xl = [f"{t[:4]}\nL{l}" for t in TYPES for l in LEVELS]
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.0))
for M in ALL:
    s = fr[fr.model == M].set_index(["type", "level"]).loc[[(t, l) for t in TYPES for l in LEVELS]]
    for k in range(3):
        sl = slice(3 * k, 3 * k + 3)
        ax[0].plot(xx[sl], s.value.values[sl], ls=LS[M], color=S.MODEL_COLORS[M], marker=S.MODEL_MARKERS[M], ms=4, lw=1.5, label=LAB[M] if k == 0 else None)
        ax[0].fill_between(xx[sl], s.ci_low.values[sl], s.ci_high.values[sl], color=S.MODEL_COLORS[M], alpha=0.15, lw=0)
group_axis(ax[0]); ax[0].set(ylabel="Share of prediction flips", ylim=(0, 0.75)); ax[0].legend(fontsize=7, loc="upper right", handlelength=3.2)
ff = r12[r12.kind != "flip_rate"].copy(); ff["method"] = ff.kind.str.replace("xfi_flip_vs_noflip_", ""); w = 0.18
for i, M in enumerate(ALL):
    for j, m in enumerate(METHODS):
        r = ff[(ff.model == M) & (ff.method == m)].iloc[0]; x0 = i + (j - 0.5) * 2 * w
        ax[1].bar([x0 - w / 2, x0 + w / 2], [r.xfi_noflip, r.xfi_flip], width=w * 0.95, color=["white", S.METHOD_COLORS[m]], edgecolor=S.METHOD_COLORS[m], lw=1.1)
ax[1].set_xticks(range(len(ALL)), [SHORT[m] for m in ALL]); ax[1].set(ylabel="Mean XFI", ylim=(0, 0.65)); ax[1].grid(axis="x", visible=False)
ax[1].legend(handles=[Patch(facecolor="white", edgecolor=S.INK, label="no flip"), Patch(facecolor=S.NEUTRAL, edgecolor=S.INK, label="flip")] +
             [Patch(facecolor=S.METHOD_COLORS[m], label=S.METHOD_LABELS[m]) for m in METHODS], fontsize=7, loc="upper left", ncol=2)
S.panel_label(ax[0], "a"); S.panel_label(ax[1], "b"); fig.subplots_adjust(wspace=0.3)
save(fig, "Fig09_prediction_flips", r12,
     "(a) Share of patients whose predicted class changes under each perturbation condition, per classifier (shaded: 95 % CI). Under Gaussian noise L2 and L3 every "
     "classifier assigns almost every image to one class (negative for Models A and B, positive for Models A-rep and A-norm), so the flip rate equals the share of patients assigned to the other class on the clean image. (b) Mean XFI of perturbed images "
     "with (filled) and without (open) a change of the predicted class, all conditions pooled.")

saved Fig09_prediction_flips


## Fig S4 · Eigen-CAM sign ablation

In [9]:
r10 = R["R10_eigen_sign"]; fig = plt.figure(figsize=(7.2, 5.6)); gs = fig.add_gridspec(2, len(ALL), height_ratios=[1, 1], hspace=0.75, wspace=0.1)
a0 = fig.add_subplot(gs[0, :]); bx = [fig.add_subplot(gs[1, k]) for k in range(len(ALL))]
for k, M in enumerate(ALL):
    s = r10[r10.model == M].set_index(["type", "level"]).loc[[(t, l) for t in TYPES for l in LEVELS]]
    for q in range(3):
        sl = slice(3 * q, 3 * q + 3)
        a0.plot(xx[sl], s.sign_flip.values[sl], ls=LS[M], color=S.MODEL_COLORS[M], marker=S.MODEL_MARKERS[M], ms=4, lw=1.5, label=LAB[M] if q == 0 else None)
        bx[k].plot(xx[sl], s.xfi_eigen.values[sl], color=S.METHOD_COLORS["EigenCAM"], marker="s", ms=3, lw=1.3)
        bx[k].plot(xx[sl], s.xfi_eigen_sc.values[sl], color=S.NEUTRAL, marker="^", ms=3, lw=1.3, ls="--")
    bx[k].set_ylim(0, 0.75); bx[k].set_xticks([1, 4, 7], ["Gauss.", "Blur", "S&P"], fontsize=7); bx[k].grid(axis="x", visible=False)
    [bx[k].axvline(v, color=S.GRID, lw=0.8) for v in (2.5, 5.5)]; bx[k].set_xlabel(LAB[M], fontsize=8)
    if k: bx[k].tick_params(labelleft=False)
group_axis(a0); a0.set(ylabel="Share of sign flips", ylim=(0, 0.45)); a0.legend(fontsize=7, loc="upper left", ncol=2, handlelength=3.2)
bx[0].set_ylabel("Mean XFI")
fig.legend(handles=[Line2D([], [], color=S.METHOD_COLORS["EigenCAM"], marker="s", label="Eigen-CAM"),
                    Line2D([], [], color=S.NEUTRAL, marker="^", ls="--", label="Eigen-CAM, sign-corrected")], fontsize=7, loc="center", ncol=2, bbox_to_anchor=(0.5, 0.47))
S.panel_label(a0, "a"); [S.panel_label(b_, l_) for b_, l_ in zip(bx, "bcde")]
save(fig, "FigS4_eigencam_sign", r10,
     "Eigen-CAM sign behaviour. (a) Share of perturbed images whose signed first-principal-component projection points opposite to that of the clean image. "
     "Line style: classifier. (b-e) Mean XFI of Eigen-CAM without and with sign correction (the projection is negated when its minimum exceeds its maximum in magnitude), "
     "three levels per perturbation type.")

saved FigS4_eigencam_sign


## Fig 10 and Fig S5 · Representative heatmaps

Patients are chosen by a fixed rule, not by eye: among the true-positive patients whose maps were stored (Phase 3, balanced sample), the three whose Grad-CAM XFI under Gaussian noise L3 is closest to the median of that sample. Each row: input with the lesion box, then Grad-CAM and Eigen-CAM on the clean image, under Gaussian noise L3 and under salt-and-pepper L3 (heatmap overlaid on the corresponding input). Arrows: shift of the activation centre of mass from the clean to the perturbed map. Numbers: XFI of the perturbed map.

In [10]:
sys.path.insert(0, str(ROOT / "src"))
from xfi import metrics as XM, perturb as XP
CONDS = XP.conditions(); IDX = {c: i + 1 for i, c in enumerate(CONDS)}         # 0 = clean
SHOW = [("clean", 0), ("Gaussian L3", IDX[("Gaussian", 3)]), ("Salt-and-pepper L3", IDX[("SaltPepper", 3)])]

def qualitative(M, stem, n=3):
    mdir = P3 / f"xfi_{M}" / "maps"
    res = pd.read_csv(P3 / f"xfi_{M}" / "results_main.csv", usecols=["Patient", "Label", "pred_clean", "Method", "Type", "Level", "xfi"])
    saved = {p.stem for p in mdir.glob("*.npz")}
    tp = res[(res.Label == 1) & (res.pred_clean == 1) & (res.Patient.isin(saved)) & (res.Method == "GradCAM") & (res.Type == "Gaussian") & (res.Level == 3)]
    if tp.empty: print(f"{M}: no stored TP maps"); return
    med = tp.xfi.median(); pick = tp.assign(d=(tp.xfi - med).abs()).nsmallest(n, "d").Patient.tolist()
    fig, ax = plt.subplots(n, 7, figsize=(7.2, 1.15 * n + 0.6), squeeze=False); rows = []
    for i, pid in enumerate(pick):
        z = np.load(mdir / f"{pid}.npz"); clean = z["clean"] / 255.0; noisy = z["noisy"].astype(np.float32); boxes = z["boxes"]
        imgs = [clean] + [noisy[j - 1] for j in range(1, 10)]
        a = ax[i, 0]; a.imshow(clean, cmap="gray", vmin=0, vmax=1)
        for x0, y0, w, h in boxes * 224 / 1024: a.add_patch(Rectangle((x0, y0), w, h, fill=False, ec="#2a78d6", lw=1))
        col = 1
        for m in ["GradCAM", "EigenCAM"]:
            maps = z[f"map_{m}"].astype(np.float32); c_com = XM.center_of_mass(maps[0])
            for lab, j in SHOW:
                a = ax[i, col]; a.imshow(imgs[j], cmap="gray", vmin=0, vmax=1); a.imshow(XM.normalize(maps[j]), cmap="inferno", alpha=0.5, vmin=0, vmax=1)
                for x0, y0, w, h in boxes * 224 / 1024: a.add_patch(Rectangle((x0, y0), w, h, fill=False, ec="#2a78d6", lw=0.8))
                if j:
                    n_com = XM.center_of_mass(maps[j]); xv = XM.xfi(maps[0], maps[j])["xfi"]
                    a.annotate("", xy=(n_com[1], n_com[0]), xytext=(c_com[1], c_com[0]), arrowprops=dict(arrowstyle="->", color="white", lw=1.2))
                    a.text(4, 218, f"{xv:.2f}", color="white", fontsize=6.5, va="bottom")
                    rows.append(dict(model=M, patient=pid, method=m, condition=lab, xfi=xv, com_clean_y=c_com[0], com_clean_x=c_com[1], com_noisy_y=n_com[0], com_noisy_x=n_com[1]))
                col += 1
        for a in ax[i]: a.set_xticks([]); a.set_yticks([]); a.grid(False); [s_.set_visible(False) for s_ in a.spines.values()]
    heads = ["Input", "Grad-CAM\nclean", "Grad-CAM\nGaussian L3", "Grad-CAM\nS&P L3", "Eigen-CAM\nclean", "Eigen-CAM\nGaussian L3", "Eigen-CAM\nS&P L3"]
    for j, hd in enumerate(heads): ax[-1, j].set_xlabel(hd, fontsize=6.5)
    for i in range(n): S.panel_label(ax[i, 3], "abcdefgh"[i])
    fig.subplots_adjust(wspace=0.03, hspace=0.45)
    save(fig, stem, pd.DataFrame(rows),
         f"Representative explanations of {LAB[M]} for true-positive patients (selection rule: Grad-CAM XFI under Gaussian noise L3 closest to the median of the stored "
         "true-positive sample). Columns: input with the radiologist's bounding box (blue), then Grad-CAM and Eigen-CAM on the clean image, under Gaussian noise L3 and under "
         "salt-and-pepper noise L3, overlaid on the corresponding input. Arrows: shift of the activation centre of mass from the clean to the perturbed map; numbers: XFI.")

qualitative("A", "Fig10_examples_modelA")
if "B" in ALL: qualitative("B", "FigS5_examples_modelB")

saved Fig10_examples_modelA


saved FigS5_examples_modelB


## Fig 11 · Activation centre-of-mass drift

In [11]:
fig, ax = plt.subplots(1, len(MAIN), figsize=(7.2, 3.0), sharey=True, squeeze=False); ax = ax[0]; d11 = []
for a, M, l in zip(ax, MAIN, "ab"):
    x = pd.read_csv(P3 / f"xfi_{M}" / "results_main.csv", usecols=["Method", "Type", "Level", "com_drift_px"]); x = x[x.Method.isin(METHODS)]
    pos = 0; ticks = []
    for t in TYPES:
        for m in METHODS:
            for lv in LEVELS:
                v = x[(x.Method == m) & (x.Type == t) & (x.Level == lv)].com_drift_px.dropna().values
                bp = a.boxplot(v, positions=[pos], widths=0.7, showfliers=False, patch_artist=True)
                [p.set(facecolor="white", edgecolor=S.METHOD_COLORS[m], lw=1.0) for p in bp["boxes"]]
                [p.set(color=S.METHOD_COLORS[m], lw=0.9) for k in ["whiskers", "caps", "medians"] for p in bp[k]]
                d11.append(dict(model=M, method=m, type=t, level=lv, n=len(v), median=np.median(v), q1=np.quantile(v, .25), q3=np.quantile(v, .75), mean=v.mean()))
                pos += 1
            pos += 0.4
        ticks.append(pos - 3.8); pos += 1
    a.set_xticks(ticks, [{"Gaussian": "Gaussian\nnoise", "Blur": "Gaussian\nblur", "SaltPepper": "Salt-and-\npepper"}[t] for t in TYPES], fontsize=7); a.grid(axis="x", visible=False)
    a.set_xlabel(f"{LAB[M]} (levels L1-L3 within each group)"); S.panel_label(a, l)
ax[0].set_ylabel("Centre-of-mass shift (pixels, 224 x 224)"); ax[0].set_ylim(0, None)
fig.legend(handles=[Patch(facecolor="white", edgecolor=S.METHOD_COLORS[m], label=S.METHOD_LABELS[m]) for m in METHODS], loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.04), fontsize=7.5)
fig.subplots_adjust(wspace=0.08)
save(fig, "Fig11_com_drift", pd.DataFrame(d11),
     "Shift of the activation centre of mass between the clean and the perturbed explanation (pixels on the 224 x 224 grid), per perturbation type and level "
     "(three boxes per method, L1 to L3 from left to right). Boxes: interquartile range and median; whiskers: 1.5 IQR.")

saved Fig11_com_drift


## Fig 1 · Framework diagram (draw.io) and caption file

Writes `Fig01_framework.drawio` (open in draw.io / diagrams.net, adjust if needed, export PNG and PDF at 300 dpi) and `captions.md` with the draft caption of every figure.

In [12]:
import subprocess
subprocess.run([sys.executable, str(ROOT / "scripts" / "fig01_framework.py"), str(FIG)], check=True)
CAPTIONS["Fig01_framework"] = ("Evaluation framework. Every test image and its nine perturbed versions are classified; explanations are computed for the class predicted on the "
    "clean image and compared between the clean and each perturbed version with the XAI Fragility Index (XFI), with localization against the radiologist's box, "
    "with the degradation of the input, and against baselines. Statistics use the patient as the unit of analysis.")
with open(FIG / "captions.md", "w", encoding="utf-8") as f:
    f.write("# Figure captions (draft, v2 data)\n\n")
    for k in sorted(CAPTIONS):
        f.write(f"## {k}\n\n{CAPTIONS[k]}\n\n")
print(sorted(p.name for p in FIG.iterdir()))

ok
['Fig01_framework.drawio', 'Fig01_framework.drawio:Zone.Identifier', 'Fig01_framework.png', 'Fig01_framework.png:Zone.Identifier', 'Fig01_framework_preview.png', 'Fig02_classifiers.pdf', 'Fig02_classifiers.png', 'Fig02_classifiers__epochs.csv', 'Fig02_classifiers__reliability.csv', 'Fig02_classifiers__roc.csv', 'Fig03_kinetics.csv', 'Fig03_kinetics.pdf', 'Fig03_kinetics.png', 'Fig04_grad_vs_eigen.csv', 'Fig04_grad_vs_eigen.pdf', 'Fig04_grad_vs_eigen.png', 'Fig05_localization.csv', 'Fig05_localization.pdf', 'Fig05_localization.png', 'Fig06_background_activation.csv', 'Fig06_background_activation.pdf', 'Fig06_background_activation.png', 'Fig07_validity.pdf', 'Fig07_validity.png', 'Fig07_validity__psnr_bins.csv', 'Fig07_validity__xfi_deciles.csv', 'Fig08_baselines.pdf', 'Fig08_baselines.png', 'Fig08_baselines__baselines.csv', 'Fig08_baselines__methods.csv', 'Fig09_prediction_flips.csv', 'Fig09_prediction_flips.pdf', 'Fig09_prediction_flips.png', 'Fig10_examples_modelA.csv', 'Fig10_exam